In [78]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

import pickle
import os

In [79]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [80]:
df = pd.read_csv(
    "/content/drive/MyDrive/AI_Personal_Stylist/datasets/body_features_with_metadata.csv"
)

print(df.shape)

df.head()

(5820, 54)


,image_name,height,shoulder_width,hip_width,torso_length,left_upper_arm,right_upper_arm,left_forearm,right_forearm,left_arm,...,body_shape,confidence,item_id,category_id,category_name,style,scale,viewpoint,occlusion,bbox
0,000242.jpg,1.393440,0.180116,0.113132,0.214008,0.246414,0.178688,0.196012,0.148303,0.442426,...,Inverted Triangle,0.70,item2,8,trousers,0,1,2,2,"[225, 737, 585, 936]"
1,000242.jpg,1.393440,0.180116,0.113132,0.214008,0.246414,0.178688,0.196012,0.148303,0.442426,...,Inverted Triangle,0.70,item1,2,long sleeve top,1,2,2,1,"[258, 424, 635, 818]"
2,127774.jpg,2.279049,0.169809,0.091758,0.226928,0.118676,0.186199,0.203284,0.335108,0.321960,...,Inverted Triangle,0.85,item2,8,trousers,0,2,2,2,"[139, 507, 352, 700]"
3,127774.jpg,2.279049,0.169809,0.091758,0.226928,0.118676,0.186199,0.203284,0.335108,0.321960,...,Inverted Triangle,0.85,item1,5,vest,3,2,2,3,"[113, 239, 362, 546]"
4,059415.jpg,1.356195,0.247312,0.161482,0.253328,0.114053,0.103921,0.200337,0.210656,0.314390,...,Inverted Triangle,0.70,item1,10,short sleeve dress,1,2,2,2,"[407, 342, 610, 728]"


In [81]:
df.info()
df.head()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5820 entries, 0 to 5819
Data columns (total 54 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   image_name             5820 non-null   object 
 1   height                 5820 non-null   float64
 2   shoulder_width         5820 non-null   float64
 3   hip_width              5820 non-null   float64
 4   torso_length           5820 non-null   float64
 5   left_upper_arm         5820 non-null   float64
 6   right_upper_arm        5820 non-null   float64
 7   left_forearm           5820 non-null   float64
 8   right_forearm          5820 non-null   float64
 9   left_arm               5820 non-null   float64
 10  right_arm              5820 non-null   float64
 11  left_thigh             5820 non-null   float64
 12  right_thigh            5820 non-null   float64
 13  left_calf              5820 non-null   float64
 14  right_calf             5820 non-null   float64
 15  left

,0
image_name,0
height,0
shoulder_width,0
hip_width,0
torso_length,0
left_upper_arm,0
right_upper_arm,0
left_forearm,0
right_forearm,0
left_arm,0


In [82]:
duplicate_count = df.drop(columns=["bbox"]).duplicated().sum()
print("Duplicates:", duplicate_count)

Duplicates: 0


In [83]:
drop_cols = [
    "bbox"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [84]:
encoder = LabelEncoder()

df["category_label"] = encoder.fit_transform(
    df["category_name"]
)

print(df[["category_name","category_label"]].head())

        category_name  category_label
0            trousers              10
1     long sleeve top               2
2            trousers              10
3                vest              11
4  short sleeve dress               3


In [85]:
os.makedirs(
    "/content/drive/MyDrive/AI_Personal_Stylist/models",
    exist_ok=True
)

pickle.dump(
    encoder,
    open(
        "/content/drive/MyDrive/AI_Personal_Stylist/models/category_encoder.pkl",
        "wb"
    )
)

In [86]:
exclude = [
    "image_name",
    "item_id",
    "category_id",
    "category_name",
    "style",
    "scale",
    "viewpoint",
    "occlusion",
    "bbox",
    "category_label",
    "body_shape",
    "text_description",
    "confidence"      # <-- add this
]

feature_columns = [

    c for c in df.columns

    if c not in exclude

]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 43
['height', 'shoulder_width', 'hip_width', 'torso_length', 'left_upper_arm', 'right_upper_arm', 'left_forearm', 'right_forearm', 'left_arm', 'right_arm', 'left_thigh', 'right_thigh', 'left_calf', 'right_calf', 'left_leg', 'right_leg', 'shoulder_hip_ratio', 'torso_leg_ratio', 'shoulder_height_ratio', 'hip_height_ratio', 'torso_height_ratio', 'arm_height_ratio', 'leg_height_ratio', 'arm_leg_ratio', 'shoulder_arm_ratio', 'hip_leg_ratio', 'left_right_arm_ratio', 'left_right_leg_ratio', 'arm_symmetry', 'leg_symmetry', 'shoulder_level_diff', 'hip_level_diff', 'knee_level_diff', 'ankle_level_diff', 'left_elbow_angle', 'right_elbow_angle', 'left_knee_angle', 'right_knee_angle', 'avg_visibility', 'min_visibility', 'max_visibility', 'visible_landmarks', 'pose_score']


In [87]:
X = df[feature_columns]

training_features = feature_columns.copy()

print("Training feature count:", len(training_features))

X = X.values

print(X.shape)

Training feature count: 43
(5820, 43)


In [88]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(5820, 43)


In [89]:
metadata = df[

    [

        "image_name",

        "category_name",

        "style",

        "viewpoint",

        "occlusion",

        "scale"

    ]

].reset_index(drop=True)

metadata.head()

,image_name,category_name,style,viewpoint,occlusion,scale
0,000242.jpg,trousers,0,2,2,1
1,000242.jpg,long sleeve top,1,2,1,2
2,127774.jpg,trousers,0,2,2,2
3,127774.jpg,vest,3,2,3,2
4,059415.jpg,short sleeve dress,1,2,2,2


In [90]:
!pip -q install faiss-cpu

In [91]:
import faiss
from sklearn.decomposition import PCA

In [92]:
pca = PCA(n_components=0.95, random_state=42)

X_pca = pca.fit_transform(X_scaled)

print("Original Shape :", X_scaled.shape)
print("Reduced Shape  :", X_pca.shape)

Original Shape : (5820, 43)
Reduced Shape  : (5820, 19)


In [101]:
print("Scaler expects:", scaler.n_features_in_)
print("PCA expects:", pca.n_features_in_)
print("Training features:", len(training_features))

Scaler expects: 43
PCA expects: 43
Training features: 43


In [93]:
X_faiss = X_pca.astype("float32")

In [94]:
dimension = X_faiss.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(X_faiss)

print("Vectors in Index:", index.ntotal)

Vectors in Index: 5820


In [95]:
query = X_faiss[0].reshape(1, -1)

distances, indices = index.search(query, 5)

print("Nearest Neighbours:")
print(indices)

Nearest Neighbours:
[[   0    1 1166 1167 5714]]


In [96]:
import joblib
joblib.dump(
    scaler,
    "/content/drive/MyDrive/AI_Personal_Stylist/models/scaler.pkl"
)

joblib.dump(
    pca,
    "/content/drive/MyDrive/AI_Personal_Stylist/models/pca.pkl"
)

joblib.dump(
    encoder,
    "/content/drive/MyDrive/AI_Personal_Stylist/models/category_encoder.pkl"
)

joblib.dump(
    metadata,
    "/content/drive/MyDrive/AI_Personal_Stylist/models/metadata.pkl"
)

joblib.dump(
    training_features,
    "/content/drive/MyDrive/AI_Personal_Stylist/models/training_features.pkl"
)

faiss.write_index(
    index,
    "/content/drive/MyDrive/AI_Personal_Stylist/models/faiss_index.bin"
)

print("Models saved successfully!")

Models saved successfully!


In [97]:
df["text_description"] = (
    "Category: " + df["category_name"].astype(str) +
    ", Style: " + df["style"].astype(str) +
    ", Viewpoint: " + df["viewpoint"].astype(str) +
    ", Occlusion: " + df["occlusion"].astype(str)
)

df[["category_name", "text_description"]].head()

,category_name,text_description
0,trousers,"Category: trousers, Style: 0, Viewpoint: 2, Oc..."
1,long sleeve top,"Category: long sleeve top, Style: 1, Viewpoint..."
2,trousers,"Category: trousers, Style: 0, Viewpoint: 2, Oc..."
3,vest,"Category: vest, Style: 3, Viewpoint: 2, Occlus..."
4,short sleeve dress,"Category: short sleeve dress, Style: 1, Viewpo..."


In [98]:
df.to_csv(
    "/content/drive/MyDrive/AI_Personal_Stylist/datasets/knowledge_base.csv",
    index=False
)

print("Knowledge base saved successfully!")

Knowledge base saved successfully!


In [99]:
query_index = 0

distances, indices = index.search(
    X_faiss[query_index].reshape(1, -1),
    5
)

results = metadata.iloc[indices[0]]

results

,image_name,category_name,style,viewpoint,occlusion,scale
0,000242.jpg,trousers,0,2,2,1
1,000242.jpg,long sleeve top,1,2,1,2
1166,095932.jpg,trousers,1,2,2,2
1167,095932.jpg,short sleeve top,2,2,1,2
5714,107929.jpg,shorts,0,2,3,2


In [100]:
for i, idx in enumerate(indices[0]):

    print("=" * 60)

    print("Rank:", i + 1)

    print(metadata.iloc[idx])

    print()

Rank: 1
image_name       000242.jpg
category_name      trousers
style                     0
viewpoint                 2
occlusion                 2
scale                     1
Name: 0, dtype: object

Rank: 2
image_name            000242.jpg
category_name    long sleeve top
style                          1
viewpoint                      2
occlusion                      1
scale                          2
Name: 1, dtype: object

Rank: 3
image_name       095932.jpg
category_name      trousers
style                     1
viewpoint                 2
occlusion                 2
scale                     2
Name: 1166, dtype: object

Rank: 4
image_name             095932.jpg
category_name    short sleeve top
style                           2
viewpoint                       2
occlusion                       1
scale                           2
Name: 1167, dtype: object

Rank: 5
image_name       107929.jpg
category_name        shorts
style                     0
viewpoint                 2
occlusio